# Page View 로그 SQL 생성 (최근 30일 추가분)
- pageName: 홈, 로그인, 회원가입, 상품목록, 장바구니, 주문결제, 주문완료, 마이페이지, 회원탈퇴
- 회원탈퇴 페이지는 정확히 WITHDRAWAL_COUNT(5)건 고정, 나머지는 8개 페이지 균등 분배
- 날짜 범위: 2026-05-18 ~ 2026-06-16 (2026-06-17 이전 기준 최근 30일)
- 시간대: 0~23시 완전 랜덤 (추가 가중치 없음)
- ROW_COUNT: 1000
- user_id: 1~100 (로그인) / null (비로그인)
- user_login_id: user0001~user0100 (로그인) / null (비로그인)
- client_uuid: 세션마다 고유 UUID

In [7]:
import random
import json
import uuid
from datetime import datetime, timedelta

In [8]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
PAGE_NAMES        = ['홈', '로그인', '회원가입', '상품목록', '장바구니', '주문결제', '주문완료', '마이페이지']
WITHDRAWAL_PAGE   = '회원탈퇴'
WITHDRAWAL_COUNT  = 5     # 회원탈퇴 페이지 로그 개수 (고정)

# 최근 30일 (2026-06-17 이전 기준 -> 2026-05-18 ~ 2026-06-16)
START_DATE = datetime(2026, 5, 18, 0, 0, 0)
END_DATE   = datetime(2026, 6, 16, 23, 59, 59)
ROW_COUNT  = 1000   # 생성할 로그 수

# 비로그인 사용자 비율 (0.0 ~ 1.0)
ANONYMOUS_RATIO = 0.3

In [9]:
def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

def format_kst(dt):
    """event_timestamp 포맷 (KST +09:00)"""
    return dt.strftime('%Y-%m-%dT%H:%M:%S.') + f"{dt.microsecond // 1000:03d}+09:00"

def format_history_ts(dt):
    """history_timestamp 포맷 (마이크로초 포함)"""
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')

In [10]:
rows = []

for i in range(ROW_COUNT):
    # event_timestamp 기준으로 먼저 뽑고, history_timestamp = event_ts + 1초 (서버 수신이 더 늦음)
    # 시간대는 0~23시 완전 랜덤 (random_datetime이 초 단위 균등 분포)
    event_ts   = random_datetime(START_DATE, END_DATE)
    history_ts = event_ts + timedelta(seconds=1)

    # 회원탈퇴 페이지는 정확히 WITHDRAWAL_COUNT건, 나머지는 8개 페이지 균등 분배
    if i < WITHDRAWAL_COUNT:
        page_name = WITHDRAWAL_PAGE
    else:
        page_name = random.choice(PAGE_NAMES)

    dwell_time = random.randint(1, 30)

    # 세션마다 고유 UUID (비로그인 사용자도 UUID는 항상 존재)
    client_uuid = str(uuid.uuid4())

    # 비로그인 사용자 여부
    is_anonymous = random.random() < ANONYMOUS_RATIO

    if is_anonymous:
        user_id       = None
        user_login_id = None
    else:
        user_id       = random.randint(1, 100)
        user_login_id = f'user{user_id:04d}'

    json_log = json.dumps({
        'event_name':      'page_view',
        'pageName':        page_name,
        'dwellTime':       dwell_time,
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(event_ts)
    }, ensure_ascii=False)

    rows.append((history_ts, json_log))

# 회원탈퇴 로그가 시간순으로 앞쪽에만 몰리지 않도록 셔플
random.shuffle(rows)

print(f'✅ {ROW_COUNT}개 page_view 로그 생성 완료 (회원탈퇴 {WITHDRAWAL_COUNT}건 포함)')

✅ 1000개 page_view 로그 생성 완료 (회원탈퇴 5건 포함)


In [11]:
# SQL 생성 및 저장
lines  = ["INSERT INTO first_save_history (history_timestamp, json_log) VALUES"]
values = []

for history_ts, json_log in rows:
    ts_str  = format_history_ts(history_ts)
    escaped = json_log.replace("'", "''")
    values.append(f"  ('{ts_str}', '{escaped}')")

lines.append(',\n'.join(values) + ';')
sql = '\n'.join(lines)

with open('page_view_logs_recent30d.sql', 'w', encoding='utf-8') as f:
    f.write(sql)

print(f'✅ {ROW_COUNT}개 page_view 로그 SQL 생성 완료 → page_view_logs_recent30d.sql')

✅ 1000개 page_view 로그 SQL 생성 완료 → page_view_logs_recent30d.sql


In [12]:
# ── 미리보기 ──
print('=== PAGE VIEW SQL (앞 500자) ===')
print(sql[:500])

=== PAGE VIEW SQL (앞 500자) ===
INSERT INTO first_save_history (history_timestamp, json_log) VALUES
  ('2026-06-14 16:32:38.000000', '{"event_name": "page_view", "pageName": "마이페이지", "dwellTime": 29, "user_id": 67, "user_login_id": "user0067", "client_uuid": "5390d67d-6938-4fa1-aec9-74a588fd218c", "event_timestamp": "2026-06-14T16:32:37.000+09:00"}'),
  ('2026-05-31 20:08:47.000000', '{"event_name": "page_view", "pageName": "주문완료", "dwellTime": 5, "user_id": 39, "user_login_id": "user0039", "client_uuid": "eff8b058-689a-41b5-9
